<a href="https://colab.research.google.com/github/vubaominh/lab1_354/blob/main/COMP333_final_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/zedex26/COMP333_FinalProject_GroupF/blob/main/COMP333_final_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project COMP333

Sarah Malik (#40175868)  
Vu Bao Minh Nguyen (#40236101)  
Xia Zhu (#26866883)

## A. Division of Labor:

???

## B. References:




Moosavi S (accessed 2026). *US Accidents (2016 - 2023).* Kaggle, https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents/data.

## Phase 1: Data acquisition, exploration, baseline analysis

Comment: Report: Jupyter notebook with code, output, visualizations, Markdown explanations, division-of-labour, references.

- Task:
+ Acquire and explore dataset
+ Identify data quality issues
+ Establish baseline performance
+ Make sure compatibility with next tasks (i.e. suitable for supervised +
unsupervised machine learning).


Deliverables:
1. **Data Retrieval (10)**: Document sources; programmatic retrieval (SQL/API/scraping); handle challenges (rate limits,auth); store raw data.
2. **Wrangling/Cleaning (10)**: Initial audit (missing values, duplicates, outliers); reproducible pipeline; use *quantDDA()*/*vizDDA()*.
3. **EDA (10)**: Summary statistics; uni/bivariate visualizations; correlation analysis; formulate 2 research questions (1 supervised, 1 unsupervised).
4. **Baseline Model (12)**: Simple model (logistic/linear regression, decision tree); train/val/test split (70%/15%/15% by default); evaluate with appropriate metrics; discuss
performance.
5. **Report (8)**: Jupyter notebook with code, output, visualizations, Markdown explanations, division-of-labour, references.


### Initialization

Install required Python libraries.

In [1]:
%pip install -q polars pyarrow kagglehub pandas matplotlib seaborn scipy

Import required packages.

In [2]:
from pathlib import Path

import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import kagglehub
except ImportError:
    kagglehub = None

Import dataset from Kaggle and checking validity.

In [3]:
# Set the file and dataset names
file_name = "US_Accidents_March23.csv"
dataset_name = "sobhanmoosavi/us-accidents"

# Prefer a local file if available
local_candidates = [
    Path.cwd() / file_name,
    Path(file_name)
]
existing_local = next((p for p in local_candidates if p.exists()), None)

if existing_local is not None:
    file_path = str(existing_local)
    print(f"Using local dataset file: {file_path}")
elif kagglehub is not None:
    local_dataset_path = kagglehub.dataset_download(dataset_name)
    file_path = str(Path(local_dataset_path) / file_name)
    print(f"Dataset downloaded to: {local_dataset_path}")
    print(f"Attempting to load file: {file_path}")
else:
    raise FileNotFoundError(
        f"Could not find '{file_name}' locally and 'kagglehub' is not available. "
        "Place the CSV beside the notebook or install kagglehub."
    )

# Load dataset into Polars LazyFrame then collect to DataFrame
lazy_df = pl.scan_csv(
    file_path,
    try_parse_dates=True,
    infer_schema_length=10000,
    low_memory=True
)

df = lazy_df.collect()
print("Polars DataFrame loaded successfully.")

Using Colab cache for faster access to the 'us-accidents' dataset.
Dataset downloaded to: /kaggle/input/us-accidents
Attempting to load file: /kaggle/input/us-accidents/US_Accidents_March23.csv
Polars DataFrame loaded successfully.


Load Schemas to make sure all the varaibles are present.

In [4]:
print("Schema:")
for col, dtype in df.schema.items():
    print(f"- {col}: {dtype}")

print("\nPreview:")
with pl.Config(tbl_cols=-1, tbl_width_chars=2000):
    print(df.head(5))

rows, cols = df.shape
print(f"\nDataset has {rows} rows and {cols} columns.")

Schema:
- ID: String
- Source: String
- Severity: Int64
- Start_Time: Datetime(time_unit='us', time_zone=None)
- End_Time: Datetime(time_unit='us', time_zone=None)
- Start_Lat: Float64
- Start_Lng: Float64
- End_Lat: String
- End_Lng: String
- Distance(mi): Float64
- Description: String
- Street: String
- City: String
- County: String
- State: String
- Zipcode: String
- Country: String
- Timezone: String
- Airport_Code: String
- Weather_Timestamp: Datetime(time_unit='us', time_zone=None)
- Temperature(F): Float64
- Wind_Chill(F): Float64
- Humidity(%): Float64
- Pressure(in): Float64
- Visibility(mi): Float64
- Wind_Direction: String
- Wind_Speed(mph): Float64
- Precipitation(in): Float64
- Weather_Condition: String
- Amenity: Boolean
- Bump: Boolean
- Crossing: Boolean
- Give_Way: Boolean
- Junction: Boolean
- No_Exit: Boolean
- Railway: Boolean
- Roundabout: Boolean
- Station: Boolean
- Stop: Boolean
- Traffic_Calming: Boolean
- Traffic_Signal: Boolean
- Turning_Loop: Boolean
- Sunri

Initialize `quantDDA(df)` function to perform Quantitative Descriptive Data Analysis on the dataset variables.

In [5]:
import polars.selectors as cs

def quantDDA(df: pl.DataFrame):
    """
    Perform Quantitative Descriptive Data Analysis on a Polars dataframe.
    """
    print("=" * 80)
    print("QUANTITATIVE DESCRIPTIVE DATA ANALYSIS (quantDDA)")
    print("=" * 80)

    total_rows, total_cols = df.shape
    all_cols = df.columns

    exclude_lower = {"description", "id"}
    exclude_cols = {c for c in all_cols if c.lower() in exclude_lower}
    analysis_cols = [c for c in all_cols if c not in exclude_cols]

    # 1. Dataset shape
    print("\n1. DATASET OVERVIEW")
    print(f"   Number of Observations (Rows): {total_rows}")
    print(f"   Number of Variables (Columns): {total_cols}")
    print(f"   Total Data Points: {total_rows * total_cols}")

    # 2. Column information
    print("\n2. COLUMN INFORMATION")
    print(f"   {'Column':<20} {'Type':<12} {'Entries':<10} {'Unique':<10} {'Missing':<10} {'Missing %':<12}")
    print("   " + "-" * 80)

    null_counts = df.null_count().row(0, named=True)
    for col_name in analysis_cols:
        dtype = df.schema[col_name]
        missing = int(null_counts.get(col_name, 0))
        entries = total_rows - missing
        unique = df.select(pl.col(col_name).n_unique()).item()
        missing_pct = (missing / total_rows * 100) if total_rows else 0
        print(f"   {col_name:<20} {str(dtype):<12} {entries:<10} {unique:<10} {missing:<10} {missing_pct:<12.2f}")

    # 3. Descriptive statistics for numeric columns
    print("\n3. DESCRIPTIVE STATISTICS (NUMERIC COLUMNS)")
    numeric_cols = [c for c in df.select(cs.numeric()).columns if c in analysis_cols]

    if numeric_cols:
        for col_name in numeric_cols:
            print(f"\n   {col_name.upper()}")
            print("   " + "-" * 70)

            series = df.get_column(col_name)
            data = series.drop_nulls().to_numpy()

            if data.size == 0:
                print("   No valid data available for this column.")
                continue

            # Basic counts
            print(f"   Observations:        {total_rows}")
            print(f"   Valid Entries:       {len(data)}")
            print(f"   Unique Values:       {series.n_unique()}")
            missing = total_rows - len(data)
            missing_pct = (missing / total_rows * 100) if total_rows else 0
            print(f"   Missing Entries:     {missing} ({missing_pct:.2f}%)")

            mean_val = float(series.mean())
            std_val = float(series.std())
            min_val = float(series.min())
            max_val = float(series.max())

            print(f"\n   Mean:                {mean_val:.4f}")
            print(f"   Std Deviation:       {std_val:.4f}")
            print(f"   Minimum:             {min_val:.4f}")
            print(f"   Maximum:             {max_val:.4f}")
            print(f"   Range:               {max_val - min_val:.4f}")

            q1 = float(series.quantile(0.25))
            q2 = float(series.quantile(0.50))
            q3 = float(series.quantile(0.75))
            print(f"\n   Q1 (25th percentile): {q1:.4f}")
            print(f"   Q2 (Median):          {q2:.4f}")
            print(f"   Q3 (75th percentile): {q3:.4f}")
            print(f"   IQR (Q3-Q1):          {q3 - q1:.4f}")

            # Mode
            value_counts = (
                df.select(pl.col(col_name)).drop_nulls()
                .group_by(col_name).len()
                .sort("len", descending=True)
            )
            if value_counts.height > 0:
                top_modes = value_counts.head(5).to_dicts()
                if len(top_modes) == 1:
                    print(f"   Mode:                 {top_modes[0][col_name]:.4f}")
                else:
                    mode_str = ", ".join([f"{row[col_name]:.4f}" for row in top_modes])
                    if value_counts.height > 5:
                        mode_str += f" ... ({value_counts.height} modes)"
                    print(f"   Modes:                {mode_str}")

            # Outliers using IQR
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            outliers = data[(data < lower_bound) | (data > upper_bound)]
            outlier_pct = (len(outliers) / len(data) * 100) if len(data) else 0
            print(f"\n   Outliers (IQR):       {len(outliers)} ({outlier_pct:.2f}%)")
            if len(outliers) > 0:
                print(f"   IQR Lower Bound:      {lower_bound:.4f}")
                print(f"   IQR Upper Bound:      {upper_bound:.4f}")

            # Extreme values
            bottom_1pct = float(series.quantile(0.01))
            top_1pct = float(series.quantile(0.99))
            extreme_low = data[data <= bottom_1pct]
            extreme_high = data[data >= top_1pct]
            print(f"\n   Bottom 1% Threshold:  {bottom_1pct:.4f} ({len(extreme_low)} values)")
            print(f"   Top 1% Threshold:     {top_1pct:.4f} ({len(extreme_high)} values)")

            # Distribution shape
            skewness = float(stats.skew(data))
            kurtosis_val = float(stats.kurtosis(data))
            print(f"\n   Skewness:             {skewness:.4f}", end="")
            if skewness > 1:
                print(" (highly right-skewed)")
            elif skewness > 0.5:
                print(" (moderately right-skewed)")
            elif skewness < -1:
                print(" (highly left-skewed)")
            elif skewness < -0.5:
                print(" (moderately left-skewed)")
            else:
                print(" (approximately symmetric)")

            print(f"   Kurtosis:             {kurtosis_val:.4f}", end="")
            if kurtosis_val > 3:
                print(" (heavy-tailed)")
            elif kurtosis_val < -1:
                print(" (light-tailed)")
            else:
                print(" (normal-like)")
    else:
        print("   No numeric columns found in dataframe.")

    # 4. Categorical column summary
    print("\n\n4. CATEGORICAL COLUMN SUMMARY")
    categorical_cols = [c for c in df.select(cs.string()).columns if c in analysis_cols]

    if categorical_cols:
        for col_name in categorical_cols:
            print(f"\n   {col_name.upper()}")
            print("   " + "-" * 70)

            series = df.get_column(col_name)
            valid = series.drop_nulls()

            print(f"   Observations:        {total_rows}")
            print(f"   Valid Entries:       {valid.len()}")
            print(f"   Unique Values:       {series.n_unique()}")
            missing = total_rows - valid.len()
            missing_pct = (missing / total_rows * 100) if total_rows else 0
            print(f"   Missing Entries:     {missing} ({missing_pct:.2f}%)")

            if valid.len() > 0:
                value_counts = (
                    df.select(pl.col(col_name)).drop_nulls()
                    .group_by(col_name).len()
                    .sort("len", descending=True)
                )
                if value_counts.height > 0:
                    top_row = value_counts.row(0, named=True)
                    print(f"   Mode:                {top_row[col_name]} ({top_row['len']} occurrences)")

                print("\n   Top 5 Most Frequent Values:")
                top_rows = value_counts.head(5).to_dicts()
                valid_len = valid.len()
                for row in top_rows:
                    count = row.get("len", 0)
                    pct = (count / valid_len * 100) if valid_len else 0
                    print(f"      {row[col_name]}: {count} ({pct:.2f}%)")
    else:
        print("   No categorical columns found in dataframe.")

    print("\n" + "=" * 80 + "\n")

Perform `quantDDA(df)` on the dataset.

In [6]:
print("Quantitative Data Analysis using Polars")
quantDDA(df)

Quantitative Data Analysis using Polars
QUANTITATIVE DESCRIPTIVE DATA ANALYSIS (quantDDA)

1. DATASET OVERVIEW
   Number of Observations (Rows): 7728394
   Number of Variables (Columns): 46
   Total Data Points: 355506124

2. COLUMN INFORMATION
   Column               Type         Entries    Unique     Missing    Missing %   
   --------------------------------------------------------------------------------
   Source               String       7728394    3          0          0.00        
   Severity             Int64        7728394    4          0          0.00        
   Start_Time           Datetime(time_unit='us', time_zone=None) 7728394    5801064    0          0.00        
   End_Time             Datetime(time_unit='us', time_zone=None) 7728394    6463024    0          0.00        
   Start_Lat            Float64      7728394    2449172    0          0.00        
   Start_Lng            Float64      7728394    2493905    0          0.00        
   End_Lat              String    

## Phase 2: Advanced modeling and comparative evaluation

Deliverables:
1. **Advanced Supervised Learning (15 + 8)**: Implement ≥ 2 models (Random Forest, XGBoost, SVM, MLP/Neural Networks);
hyperparameter tuning; evaluate with multiple metrics; systematic comparison (tables, ROC curves, AUC, confusion
matrices); justify best model.
2. **Feature Engineering (8)**: New features (polynomial, interactions, domain-specific, text/time features); feature selection
(filter, wrapper, embedded methods); compare performance.
3. **Unsupervised Learning (12)**: Implement dimension reduction SVD or PCA; determine optimal clusters; visualize; evaluate quality; justify appropriateness.
4. **Interpretation (7)**: Feature importance, partial dependence plots; discuss insights; relate to research questions.
5. **Report**: Update notebook with Phase 1 (revised) + Phase 2 work; update division-of-labour.


## Phase 3: Complete pipeline and demonstration

Deliverables:
1. **End-to-End Pipeline (30)**: Integrate all phases; refactor into modular functions/classes; error handling; reproducible on
new data.
2. **Comprehensive Evaluation (25)**: Test set performance; robustness analysis; results tables/visualizations.
3. **Bonus (Optional, 10)**: Deep learning, ensemble stacking, time series, anomaly detection, NLP, REST API,
interactive dashboard.
4. **Ethical Considerations (10)**: Discuss bias, fairness, privacy; identify limitations; suggest future work.
5. **Final Report (20 + 10)**: Comprehensive notebook; executive summary (1–2 pages); table of contents; professional visualizations;
complete division-of-labour; references.
6. **Live Demonstration (50)**: Present research questions/dataset; demonstrate pipeline; highlight findings; discuss challenges; answer individual questions.